# H7 · `scripts/layers.py`

## What this file is for

The layered test programme behind the `/tests` page: four layers -- smoke, classification, limits,
deductibles -- each adding exactly one kind of variation, so a difference found at layer *n* is
attributable to what layer *n* introduced. Sits **beside** `qa.py` rather than inside it: the tier
runner is working and being used, and this is a different unit of work with a different budget
model. They share the run store, the variant definitions and the sweep.

**Every state, every run, no promotion step.** The offline pass is a free pre-flight, not a
decision -- it keeps an unbuildable payload from spending a live call, and settles nothing about who
is right; only ISO's actual response does that.

**The allowance is denominated in live calls and thins configurations, never states**, so two runs
of the same layer stay comparable at different budgets, and a thinned run says what it dropped.

**Depends on:** [`H1 variants.py`](01-variants.ipynb) for a control, [`H4 sweep.py`](04-sweep.ipynb)
to run one, [`H5 runstore.py`](05-runstore.ipynb) to record it. **This notebook redirects the store
to a temporary directory** before calling `run()`, for the same reason
[`H5`](05-runstore.ipynb) does -- a demo run must not land in the real results.

## Its public surface

Generated from the module, so it can't drift.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
sys.path.insert(0, str(Path.cwd().parent.parent / "scripts"))

import inspect
import layers as L

for name, obj in vars(L).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != L.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        v = repr(obj)
        print(f"{name} = {v if len(v) < 90 else v[:87] + '...'}")

## The smallest thing that works

`plan()` says what a run would do without doing any of it -- the same thing `--plan` prints from the
CLI, and what the `/tests` page shows before you press *Run it*.

In [ ]:
for k in sorted(L.LAYERS):
    spec = L.LAYERS[k]
    print(f"{k}  {spec['name']:<16} varies: {spec['varies']}")

print()
p = L.plan("L1", jurisdictions=["TX", "CA", "AK"])
print(f"L1 in 3 states: {p['cost']}")

## The interesting case

### Naming a fixed aggregate does not work in 51 states -- so L3 asks for a position

`AGGREGATE_POSITIONS` are `@lowest`, `@middle`, `@highest`, not figures. `resolve()` turns a
position into whatever that jurisdiction actually declares, per state, at run time.

### Thinning drops configs at the ends and an even spread, never states

Give L3 too small an allowance and it keeps the two ends of the occurrence-limit range rather than
the middle, because the ends of a filed table are where a keying error shows.

In [ ]:
p = L.plan("L3", jurisdictions=["TX", "CA", "NY"], allowance=6)
t = p["thinning"]
print(f"L3 planned {t['configs_planned']} configs, kept {t['configs_kept']} "
      f"to fit an allowance of {t['allowance']}\n")
print("kept   :", *t["kept"], sep="\n  ")
print(f"\ndropped: {len(t['dropped'])} configs, all in the middle of the range")

### The basis grouping is a guard that almost never fires

Measured across six states: about 1,188 class codes common to all of them, and the premium basis
differed for exactly one. `basis_groups` still checks every run, because the one time it fires, a
premium compared across two different bases is a units artifact, not a rating difference.

In [ ]:
g = L.basis_groups("91340", ["TX", "CA", "NY", "FL"])
print("groups     :", g["groups"])
print("undeclared :", g["undeclared"] or "(filed everywhere asked)")

### A demo run, stored nowhere real

`run()` executes a layer end to end -- offline pre-flight, then live if asked -- and records every
scenario through `runstore.append`. Pointed at a temporary directory here, exactly as
[`H5`](05-runstore.ipynb) does it.

In [ ]:
import tempfile
import runstore as store

real_results = store.RESULTS
store.RESULTS = Path(tempfile.mkdtemp())

out = L.run("L1", jurisdictions=["TX", "CA"], offline=True, label="notebook demo")
print("rollup  :", out["plan"]["rollup"])
print("run_ids :", out["plan"]["run_ids"])

store.RESULTS = real_results

## What it refuses

A layer that needs a class and was not given one, and a layer name that does not exist -- both
refuse before touching the declaration at all.

In [ ]:
try:
    L.plan("L2", jurisdictions=["TX"])
except L.PlanError as exc:
    print("REFUSED:", exc)

try:
    L.plan("L9")
except L.PlanError as exc:
    print("REFUSED:", exc)

## Try it yourself

1. `python scripts/layers.py --layer L4 --allowance 30 --plan` from a terminal -- L4 has six
   configs, one per deductible slot. Does an allowance of 30 thin anything, and why might that
   differ from L3's answer?
2. A **stopped** run: pass `stop_check` to `run()` so it returns truthy on the second scenario.
   What does `out["plan"]["stopped_after"]` say, and what happened to the scenarios after it?
3. `ui/tests_page.py` and `ui/runfile.py` are what turn this module's output into the `/tests` page
   and a standalone run file -- not covered in this set, because they render rather than compute.
   Read `runfile.write_run` next to see how a run becomes the interactive matrix.

In [ ]:
# your turn